

**Summary:** In this lesson, we won't just classify images but we'll take it a step further by detecting objects within them. We'll use a popular object detection model called YOLO (You Only Look Once).

**Objectives:**

Detect objects in an image using the YOLO model

Parse the results from running the YOLO model

Display the resulting bounding boxes from running the YOLO model

Detect objects from a variety of sources, including videos and images stored in directories

Detect objects from a video source

**New Terms:** - Object detection - YOLO - Bounding boxes - Normalized coordinates


In [ ]:
import sys
from collections import Counter
from pathlib import Path

import PIL
import torch
import torchvision
# import ultralytics
from IPython.display import Video
from PIL import Image
from torchvision import transforms
from torchvision.io import read_image
from torchvision.utils import make_grid
from ultralytics import YOLO


In [ ]:
%pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 26.7 MB/s eta 0:00:00


In [ ]:
import sys
from collections import Counter
from pathlib import Path

import PIL
import torch
import torchvision
import ultralytics
from IPython.display import Video
from PIL import Image
from torchvision import transforms
from torchvision.io import read_image
from torchvision.utils import make_grid
from ultralytics import YOLO
# import YOLO

In [ ]:
print("Platform:", sys.platform)
print("Python version:", sys.version)
print("---")
print("PIL version : ", PIL.__version__)
print("torch version : ", torch.__version__)
print("torchvision version : ", torchvision.__version__)
print("ultralytics version : ", ultralytics.__version__)

Platform: linux
Python version: 3.12.11 (main, Jun  4 2025, 08:56:18) [GCC 11.4.0]
---
PIL version :  11.3.0
torch version :  2.8.0+cu126
torchvision version :  0.23.0+cu126
ultralytics version :  8.3.202


In [ ]:
yolo = YOLO(task="detect", model="yolov8s.pt")


In [ ]:
yolo.names

{0: 'person',
 1: 'bicycle',
 2: 'car',
 3: 'motorcycle',
 4: 'airplane',
 5: 'bus',
 6: 'train',
 7: 'truck',
 8: 'boat',
 9: 'traffic light',
 10: 'fire hydrant',
 11: 'stop sign',
 12: 'parking meter',
 13: 'bench',
 14: 'bird',
 15: 'cat',
 16: 'dog',
 17: 'horse',
 18: 'sheep',
 19: 'cow',
 20: 'elephant',
 21: 'bear',
 22: 'zebra',
 23: 'giraffe',
 24: 'backpack',
 25: 'umbrella',
 26: 'handbag',
 27: 'tie',
 28: 'suitcase',
 29: 'frisbee',
 30: 'skis',
 31: 'snowboard',
 32: 'sports ball',
 33: 'kite',
 34: 'baseball bat',
 35: 'baseball glove',
 36: 'skateboard',
 37: 'surfboard',
 38: 'tennis racket',
 39: 'bottle',
 40: 'wine glass',
 41: 'cup',
 42: 'fork',
 43: 'knife',
 44: 'spoon',
 45: 'bowl',
 46: 'banana',
 47: 'apple',
 48: 'sandwich',
 49: 'orange',
 50: 'broccoli',
 51: 'carrot',
 52: 'hot dog',
 53: 'pizza',
 54: 'donut',
 55: 'cake',
 56: 'chair',
 57: 'couch',
 58: 'potted plant',
 59: 'bed',
 60: 'dining table',
 61: 'toilet',
 62: 'tv',
 63: 'laptop',
 64: 'mou

**Task 3.3.1: Determine the class that's assigned to integer 23?**

In [ ]:
class_assigned_to_23 = yolo.names[23]
print(f"{class_assigned_to_23} corresponds to 23")

giraffe corresponds to 23


Our task involves identifying objects from traffic video feeds. There are several objects we want to detect that are not included in the classes from the pretrained YOLO model. These classes are defined below.

In [ ]:
classes_not_in_yolo = [
    "ambulance",
    "army vehicle",
    "auto rickshaw",
    "garbagevan",
    "human hauler",
    "minibus",
    "minivan",
    "pickup",
    "policecar",
    "rickshaw",
    "scooter",
    "suv",
    "taxi",
    "three wheelers (CNG)",
    "van",
    "wheelbarrow",
]


Let's double check that "ambulance" is not in the YOLO classes.

In [ ]:
"ambulance" not in yolo.names.values()

True

In [ ]:
"gun" not in yolo.names.values()

True

**Task 3.3.2: Double check that "army vehicle" is not in the YOLO classes.**

In [ ]:
is_army_vehicle_inlcuded = "army vehicles" not in yolo.names.values()
print(is_army_vehicle_inlcuded)

True


In a later lesson, we'll retrain the YOLO model to include the missing classes. For this lesson, we are OK with what's already provided. We are most interested in the first 13 classes. Those classes are objects often found in traffic.

Let's use the YOLO model to identify objects in one frame of our video data. We'll use Path provided by pathlib.

**Task 3.3.3: Run the YOLO model on frame_2575.jpg.**

In [ ]:
data_dir = Path("data_video", "extracted_frames")
image_path = data_dir / "frame_1050.jpg"

result = yolo(image_path)

In [ ]:
print(f"Type of result: {type(result)}")
print(f"Length of result: {len(result)}")

It explains another way to use the YOLO model—through the .predict method. This approach makes the process clearer and gives flexibility to adjust default settings, such as setting a 50% confidence threshold for bounding boxes, and also allows saving the prediction results to a text file.

**Task 3.3.4: Use the predict method for frame_2575.jpg. Make sure you use a 50% confidence threshold and save the results as a text file.**

In [ ]:
result = yolo.predict(image_path, conf=0.5, save=True, save_txt=True)

We'll need to further unpack what's inside the .boxes attribute. The .cls attribute contains the classes of each of the objects detected. It's a PyTorch tensor. The length of the tensor is the number of objects detected.

In [ ]:
print(result[0].boxes.cls)
print(f"Number of objects detected: {len(result[0].boxes.cls)}")

**Task 3.3.5: Determine the number of detected objects in frame_2575.jpg.**

In [ ]:
number_of_detected_objs = result[0].boxes.cls
print(f"Number of objects detected in frame_2575.jpg: {number_of_detected_objs}")

**Task 3.3.6: Determine the most common class and the number of times it was detected in frame_2575.jpg.**

In [ ]:
object_counts_task = Counter([yolo.names[int(cls)] for cls in result[0].boxes.cls])

most_common_class, count_of_class = object_counts_task.most_common(n=1)[0]
print(f"Most common class: {most_common_class}")
print(f"Number of detected {most_common_class}: {count_of_class}")

Another important attribute is `.conf` which has the confidence of the detected bounding boxes. The confidence is stored in a PyTorch tensor. We should expect this tensor's length to match the number we saw earlier

**Task 3.3.7: Check the length of the confidence tensor of result_task to verify this number matches to what was observed earlier.**

In [ ]:
length_of_confidence_tensor = len(result_task[0].boxes.conf)
print(f"Number of objects detected: {length_of_confidence_tensor}")

When calling .predict, we set the confidence threshold to 50%. That is why all values in the confidence tensor is greater than 0.5. How many of the bounding boxes have a confidence value greater than 75%? For frame frame_1050.jpg, that would be

In [ ]:
number_of_confident_objects = (result[0].boxes.conf > 0.75).sum().item()
print(f"Number of objects detected with 50% confidence: {number_of_confident_objects}")

**Task 3.3.8: Calculate the number of objects that were detected in frame_2575.jpg with 75% confidence.**

In [ ]:
number_of_confident_objects_task = (result[0].boxes.conf > 0.75).sum().item()

print(
    f"Number of objects detected in frame_2575.jpg with 50% confidence: {number_of_confident_objects_task}"
)

The .data attribute has the raw detection info, but we won’t use it because YOLO already gives us simpler attributes for bounding boxes.

.orig_shape → shows the original size of the input image.

is_track → tells if object tracking is turned on (useful for videos or multiple frames).

YOLO provides four different formats of bounding box data, each using 4 values to describe a box. Different tools may need different formats.

Example:

.xywh → gives boxes as:

x, y = top-left corner of the box

w, h = width and height

In [ ]:
result[0].boxes.xywh

.xywhn is very similar to .xywh but these coordinates have been normalized by the image size. We can remind ourselves of the original shape with .orig_shape.

In [ ]:
result[0].orig_shape

This means the image is 360 pixels high and 640 pixels wide. Let's examine one row of the normalized bounding box.

Now we can use the original shape to verify that indeed .xywhn is normalized.

In [ ]:
result[0].boxes.xywh[0] / torch.Tensor([640, 360, 640, 360]).to("cuda")

T**ask 3.3.9: Print out the original shape of frame_2575.jpg.**

In [ ]:
original_shape_task = result[0].orig_shape
print(f"Original shape of frame_2574.jpg: {original_shape_task}")

**Task 3.3.10: Print out the normalized xywh bounding box for the first object of frame_2575.jpg.**

In [ ]:
normalized_xywh = result_task[0].boxes.xywh
print(f"Normalized xywh bounding box for frame_2575.jpg: {normalized_xywh[0]}")

**Task 3.3.11: Normalize the bounding box using the original shape of the frame_2575.jpg.**

In [ ]:
normalized_xywh_task = result[0].boxes.xywh[0] / torch.Tensor([640, 360, 640, 360]).to("cuda")
print(f"Normalized xywh bounding box for frame_2575.jpg: {normalized_xywh[0]}")

The third provided bounding box form is .xyxy. This form contains two coordinates, the (x, y) coordinate for the top left corner and the (x, y) coordinate of the bottom right corner.

result[0].boxes.xyxy








The last form is .xyxyn which is the normalized form of .xyxy.

In [ ]:
result[0].boxes.xyxyn

We've explored the most important attributes of the .boxes attribute of the returned result object. Now let's return the remaining important attributes. .save_dir is just the location where we've saved the resulting bounding boxes. We'll use the method exists of a Path object to make sure the location actually exists.

**Task 3.3.12: Determine the location for the results of frame_2575.jpg.**

In [ ]:
location_of_results = Path(result[0].save_dir)

print(f"Results saved to {location_of_results}")
location_of_results.exists()

In [ ]:
location_of_results_task = result_task[0].save_dir
print(f"Results for frame_2575.jpg saved to {location_of_results_task}")

Finally, .speed is a dictionary for the time it took to run the preprocessing, inference (prediction), and postprocessing steps. These times are measured in milliseconds. A good rule of thumb is that times less than 100 milliseconds are experienced as instantaneous.

In [ ]:
result[0].speed

**Task 3.3.13: Calculate the total time object detection took for frame_2575.jpg.**

In [ ]:
total_time = result_task[0].speed
print(f"Total time in milliseconds: {total_time}")

By saving our results, we've created an image file with the bounding boxes drawn in.

In [ ]:
Image.open(location_of_results / "frame_1050.jpg")

Notice how each class uses a different color, the labels are displayed, along with the confidence of the bounding box.

**Task 3.3.14: Display image frame_2575.jpg with its drawn bounding boxes.**

In [ ]:
# Display image frame_2575.jpg with the bounding boxes
location_of_results_task
Image.open(location_of_results_task / "frame_2575.jpg")

The bounding boxes were saved as a text file.

In [ ]:
with (location_of_results / "labels" / "frame_1050.txt").open("r") as f:
    print(f.read())

**Task 3.3.15: Display the text file results for the bounding box for frame_2575.jpg.**

In [ ]:
with (location_of_results_task / "labels" / "frame_1050.txt").open("r") as f:
    print(f.read())

We're ready to move on to using YOLO for identifying objects across multiple images. For convenience, we'll define a function that accepts a directory of images and displays them in a grid.

In [ ]:
def display_sample_images(dir_path, sample=5):
    dir_path = Path(dir_path) if isinstance(dir_path, str) else dir_path

    image_list = []
    # Sort the images to ensure they are processed in order
    images = sorted(dir_path.glob("*.jpg"))
    if not images:
        return None

    # Iterate over the first 'sample' images
    for img_path in images[:sample]:
        img = read_image(str(img_path))
        resize_transform = transforms.Resize((240, 240))
        img = resize_transform(img)
        image_list.append(img)

    # Organize the grid to have 'sample' images per row
    Grid = make_grid(image_list, nrow=5)
    # Convert the tensor grid to a PIL Image for display
    img = torchvision.transforms.ToPILImage()(Grid)
    return img

**Task 3.3.16: Use display_sample_images to display the first ten frames in a grid.**

In [ ]:
# Display the first ten images
display_sample_images(data_dir, sample=10)

We'll create a list of the path of 25 images from the extracted frames.

In [ ]:
images_path = list(data_dir.iterdir())[:25]
images_path

**Task 3.3.17: Create a list of the last ten frames as listed by data_dir.iterdir().**

In [ ]:
images_path_task = list(data_dir.iterdir())[:10]

print(f"Number of frames in list: {len(images_path_task)}")
images_path_task

In [ ]:
results = yolo.predict(
    images_path,
    conf=0.5,
    save=True,
    save_txt=True,
    project=Path("runs", "detect"),
    name="multiple_frames",
)

We'll once again use yolo.predict but this time we'll make use of two additional arguments to control where the results are saved. By using project and name, the saved results will be in project/name.

In [ ]:
print(results[0].save_dir)

**Task 3.3.18: Use yolo.predict on images_path_task. Save the results to runs/detect/multiple_frames_task.**

In [ ]:
results_task = yolo.predict(images_path_task,conf=0.5,save=True,save_txt=True,project=Path("runs","detect"),name="multiple_frames_task",)

print(f"\nResults from task saved to: {results_task[0].save_dir}")

In [ ]:
image = display_sample_images(results[0].save_dir, sample=25)
image

To speed things up, we're going to truncate our video and run YOLO against the truncated version. We'll use ffmpeg, a command line tool for video and audio editing. The part that controls the timestamps for truncation are the numbers that follow -ss and -to. The number after -ss is the starting timestamp and -to is the ending timestamp. The value data_video/dhaka_traffic_truncated.mp4 is the path of the created file.

In [ ]:
!ffmpeg -ss 00:00:00 -to 00:00:30 -y -i $video_path -c copy data_video/dhaka_traffic_truncated.mp4

**Task 3.3.19: Display the images with the bounding boxes with display_sample_images for the results generated in the previous task. Make sure to set sample to 10.**

In [ ]:
image_task = display_sample_images(results_task[0].save_dir,sample=10)
image_task

**Task 3.3.20: Truncate the same video as above but from the 00:00:30 to 00:01:00 timestamp and name the video data_video/dhaka_traffic_truncated_task.mp4.**

In [ ]:
image_task = display_sample_images(results_task[0].save_dir,sample=10)
image_task

In [ ]:
!ffmpeg  -ss 00:00:30 -to 00:01:00 -y -i $video_path -c copy data_video/dhaka_traffic_truncated_task.mp4

video_truncated_path_task = Path("data_video", "dhaka_traffic_truncated_task.mp4")
Video(video_truncated_path_task)

In [ ]:
results_video = yolo.predict(
    video_truncated_path,
    conf=0.5,
    save=True,
    stream=True,
    project=Path("runs", "detect"),
    name="video_source",
)

In [ ]:
for result in results_video:
    continue

YOLO will create a video of the source video with the bounding boxes. Before we display the video, we'll need to convert it to an mp4 as that format provides better compression. Better compression leads to a smaller file size. The notebook environment might have issues with playing large files. Once again, ffmpeg is the tool to use.

In [ ]:
!ffmpeg -y -i runs/detect/command_line/dhaka_traffic_truncated.avi output.mp4

**Task 3.3.21: Use YOLO in the command line for the truncated video. You'll need to change source to be $video_truncated_path_task and the name to be command_line_task.**

In [ ]:
!yolo task=detect mode=predict conf=0.5 model=yolov8s.pt source=$video_truncated_path_project project="runs/detect" name="command_line_task" > null/dev


# This will convert your video to mp4 and display it in the notebook
!ffmpeg -y -i runs/detect/command_line_task/dhaka_traffic_truncated_task.avi output_task.mp4
Video("output_task.mp4")